#  Introducción a GEN AI

👤 **Autor:** John Leonardo Vargas Mesa  
🔗 [LinkedIn](https://www.linkedin.com/in/leonardovargas/) | [GitHub](https://github.com/LeStark)  

## 📂 Repositorio en GitHub  
- 📓 **Notebooks:** [Acceder aquí](https://github.com/LeStark/Cursos/tree/main/02%20-%20GEN-AI)  
- 📑 **Data sets:** [Acceder aquí](https://github.com/LeStark/Cursos/tree/main/00%20-%20Data)  
---

### Uso básico de Amazon Bedrock con boto3

En esta sección se muestra un ejemplo mínimo y funcional de cómo consumir un modelo de Amazon Bedrock usando el SDK oficial de AWS para Python (`boto3`).  

El objetivo es ilustrar cómo establecer la conexión con Bedrock Runtime, enviar un mensaje a un modelo generativo y recuperar la respuesta en texto plano.


In [ ]:
# Se importa la librería boto3, que es el SDK oficial de AWS para Python.
# Permite autenticarse contra AWS y consumir servicios como Amazon Bedrock.
import boto3

# ¿Por qué utilizar Amazon Bedrock y no simplemente la API de algun proveedor directamente?

La principal razón para utilizar **Amazon Bedrock** en lugar de la API directa de es la **flexibilidad, la integración con el ecosistema de AWS y las capacidades de seguridad y gestión a nivel empresarial**.

### 1. Flexibilidad y Modelos Múltiples

*   **Neutralidad de modelos:** Bedrock es un servicio totalmente gestionado que ofrece acceso a una variedad de modelos fundacionales (FM) de alto rendimiento de diferentes proveedores (como Anthropic, AI21 Labs, Meta, Stability AI, y también modelos de peso abierto de OpenAI), todo a través de una API unificada. Esto permite a los usuarios cambiar de un modelo a otro (por ejemplo, de Claude a Llama 3) sin reescribir el código de integración, lo que ayuda a evitar la dependencia de un solo proveedor.
*   **Playgrounds:** Bedrock incluye "Playgrounds" que facilitan la experimentación y comparación de diferentes modelos para encontrar el que mejor se adapte a sus necesidades específicas antes de la implementación.

### 2. Integración con el Ecosistema de AWS

*   **Latencia reducida:** Si su infraestructura ya está en AWS (almacenamiento, bases de datos, etc.), el uso de Bedrock minimiza la latencia de red, ya que todos los servicios funcionan dentro de la misma infraestructura, lo que resulta en un mejor rendimiento.
*   **Gestión unificada:** Permite centralizar la gestión de recursos y el control de acceso mediante [AWS Identity and Access Management (IAM)](aws.amazon.com), simplificando la administración general del entorno en la nube.
*   **Servicios adicionales:** Bedrock se integra fácilmente con otros servicios de AWS, como Amazon S3 para almacenamiento de datos, y ofrece capacidades como "Knowledge Bases" para integrar sus propios datos y "Agents" para crear aplicaciones de IA agenciales complejas.

### 3. Seguridad y Capacidades Empresariales

*   **Gestión sin servidor:** Amazon Bedrock opera como un servicio sin servidor, lo que significa que las empresas no tienen que administrar ni mantener ninguna infraestructura subyacente, enfocándose únicamente en la creación de sus aplicaciones de IA generativa.
*   **Guardrails (Barreras de seguridad):** Ofrece protecciones configurables (Guardrails) que permiten a las organizaciones implementar IA generativa de manera segura en entornos que requieren validación rigurosa de la información y cumplimiento normativo, controlando las salidas del modelo según las políticas de la empresa.


In [ ]:


# Región de AWS donde está habilitado Amazon Bedrock.
# Debe coincidir con la región configurada en tus credenciales.
REGION = "us-east-2"

# Se crea el cliente de Bedrock Runtime.
# Este cliente es el responsable de enviar solicitudes de inferencia
# a los modelos fundacionales disponibles en Amazon Bedrock.
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

# Se realiza una llamada al método 'converse', que permite interactuar
# con modelos tipo chat de forma estructurada.
# 
# - modelId: identifica el modelo fundacional que se va a usar.
# - messages: representa el historial del diálogo en formato role/content,
#   similar a otras APIs de modelos conversacionales.
response = bedrock.converse(
    modelId="us.amazon.nova-lite-v1:0",
    messages=[
        {
            "role": "user",
            "content": [
                {"text": "Explica facilmente a los estudiantes que es Gen AI."}
            ]
        }
    ]
)

# La respuesta de Bedrock viene en una estructura JSON.
# Aquí se extrae el texto generado por el modelo desde la respuesta.
output_text = response["output"]["message"]["content"][0]["text"]

# Se imprime el resultado final generado por el modelo.
print(output_text)

In [ ]:
import json

print(json.dumps(response, indent=2, ensure_ascii=False))

### Uso de *system prompt* en Amazon Bedrock

En este ejemplo se utiliza el parámetro `system` del método `converse` para **controlar el comportamiento del modelo** antes de procesar el prompt del usuario. El *system prompt* define el rol, el tono y el nivel de profundidad con el que el modelo debe responder, independientemente de la pregunta específica.

Aquí se instruye al modelo para actuar como un **youtuber educativo**, explicando los conceptos de forma clara, con analogías simples, un tono cercano y evitando tecnicismos innecesarios. Luego, el mensaje del usuario contiene únicamente la pregunta que se desea responder.

Este patrón es fundamental en aplicaciones reales, ya que permite separar:
- **Reglas de comportamiento** (system prompt)
- **Intención puntual del usuario** (user prompt)

Gracias a esta separación, se logra mayor consistencia en las respuestas y un mejor control del estilo del modelo.


In [ ]:
response = bedrock.converse(
    modelId="us.amazon.nova-lite-v1:0",
    system=[                       # ← AQUÍ va el system prompt
        {
            "text": (
                "Eres un youtuber educativo que crea contenido para estudiantes. "
                "Explicas conceptos de forma clara, con analogías simples y divertidas, "
                "sin demasiado tecnicismo y con un tono cercano."
            )
        }
    ],
    messages=[
        {
            "role": "user",
            "content": [
                {"text": "Explica fácilmente qué es la IA Generativa."}
            ]
        }
    ]
)

# Extracción del texto generado
output_text = response["output"]["message"]["content"][0]["text"]

# Impresión de la respuesta
print(output_text)


# Demo de IA Generativa con Amazon Bedrock y Streamlit

## Forma de ejecución

Este ejemplo **no se ejecuta directamente como un notebook tradicional**.  
Debe lanzarse como una aplicación web local utilizando Streamlit desde la terminal.

Esto permite simular de forma muy cercana el comportamiento de una aplicación real que consume IA generativa.

## Arquitectura conceptual del ejemplo

El flujo del sistema es el siguiente:

1. El usuario escribe una instrucción o pregunta (prompt).
2. La aplicación envía ese texto a un modelo generativo en Amazon Bedrock.
3. El modelo procesa la solicitud y genera una respuesta.
4. La respuesta se devuelve y se muestra en la interfaz.

Este patrón es la base de chatbots, asistentes virtuales y múltiples aplicaciones basadas en LLM.

## Rol de Amazon Bedrock

Amazon Bedrock actúa como la **capa de inferencia** de modelos fundacionales.  
En este caso se utiliza un modelo conversacional liviano, ideal para demostraciones y pruebas rápidas.

Bedrock permite:
- Acceder a modelos sin gestionarlos directamente.
- Enviar prompts de manera estructurada.
- Recibir respuestas listas para ser integradas en aplicaciones.

## Interfaz de usuario (Streamlit)

La aplicación utiliza una interfaz mínima que incluye:

- Un campo de texto donde el usuario escribe el prompt.
- Un botón para enviar la solicitud.
- Un área donde se muestra la respuesta generada.

## Validaciones básicas

Antes de enviar la solicitud al modelo, se valida que el prompt no esté vacío.  
Esta práctica es importante porque:

- Evita solicitudes innecesarias al modelo.
- Mejora la experiencia del usuario.
- Reduce costos en escenarios reales de uso.

## Resultado esperado

Como salida, el usuario recibe un texto generado por el modelo que responde a su instrucción.  
El contenido de la respuesta dependerá directamente de:

- La claridad del prompt.
- El nivel de detalle solicitado.
- El tono y contexto definidos por el usuario.


In [ ]:
# NOTA:
# Este archivo debe ejecutarse desde la terminal con:
# streamlit run app.py

import streamlit as st
import boto3

# -----------------------------
# Configuración básica
# -----------------------------
REGION = "us-east-2"
MODEL_ID = "us.amazon.nova-lite-v1:0"

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

# -----------------------------
# Interfaz Streamlit
# -----------------------------
st.title("Demo simple de IA Generativa con Amazon Bedrock")
st.write(
    "Este ejemplo muestra cómo enviar un prompt a un modelo generativo "
    "y mostrar la respuesta en una aplicación web sencilla."
)

# Entrada de texto del usuario
user_prompt = st.text_area(
    "Escribe una instrucción o pregunta para el modelo:",
    placeholder="Ej: Explica qué es la IA generativa en términos simples"
)

# Botón para ejecutar la generación
if st.button("Generar respuesta"):

    if not user_prompt.strip():
        st.warning("Por favor escribe un texto antes de continuar.")
    else:
        with st.spinner("Generando respuesta..."):
            response = bedrock.converse(
                modelId=MODEL_ID,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"text": user_prompt}
                        ]
                    }
                ]
            )

            output_text = response["output"]["message"]["content"][0]["text"]

        st.subheader("Respuesta del modelo")
        st.write(output_text)


In [ ]:
import boto3
import json
import base64
from PIL import Image
from io import BytesIO

# -----------------------------
# Configuración básica
# -----------------------------
REGION = "us-east-2"
MODEL_ID = "us.amazon.titan-image-generator-v1"

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

# -----------------------------
# Prompt de prueba
# -----------------------------
prompt = "Un gato astronauta flotando en el espacio, estilo ilustración digital"

# -----------------------------
# Construir payload
# -----------------------------
body = json.dumps({
    "taskType": "TEXT_IMAGE",
    "textToImageParams": {
        "text": prompt
    },
    "imageGenerationConfig": {
        "numberOfImages": 1,
        "width": 512,
        "height": 512,
        "cfgScale": 8.0,
        "seed": 0
    }
})

# -----------------------------
# Invocar el modelo
# -----------------------------
response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=body,
    contentType="application/json",
    accept="application/json"
)

# -----------------------------
# Procesar respuesta
# -----------------------------
response_body = json.loads(response["body"].read())

# La imagen viene en base64
image_base64 = response_body["images"][0]
image_bytes = base64.b64decode(image_base64)

# Convertir a imagen y mostrar
image = Image.open(BytesIO(image_bytes))
display(image)
